# DD-03 · App process launch: Zygote fork to Application.onCreate

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/DD-03.ipynb)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: DD-03
title: "App process launch: Zygote fork to Application.onCreate"
track: android
date: 2026-10-17
code: ""
status: partial  # not-started | in-progress | partial | done
score: ""
insight: "Cold-start time is how long the main thread is busy before the first frame: Zygote forks a warm process, ActivityThread.main starts the Looper, then Application, the ContentProviders (before Application.onCreate) and the Activity all run on the main thread, so doing less there is how an app starts faster."
tags: [DD-03, android, app-launch, zygote, looper, cold-start, content-provider]
pending:
  - "Probe lab: the seven log lines that prove the order Application, providers, Activity"
  - "Perfetto trace of the cold start and the landmark table"
  - "MQ31 and MQ32 in my own words"
  - "English (overhead, redundant) and the trade-off matrix drill"
  - "Block A: DSA timed set (LC 410, LC 1482, LC 162)"
```

## Goal
Know exactly what happens between tapping an icon and the first frame, and why app launch is fast (Zygote) and where startup time is spent (the main thread).

## The story: a cold start in 9 steps
```text
1  You tap the icon
2  Launcher asks system_server to start the app            (a Binder call)
3  system_server finds NO process → shows a placeholder window (splash) at once
4  system_server asks Zygote for a process                 (the Zygote socket, DD-02)
5  The child runs ActivityThread.main()                    → Looper.prepareMainLooper, attach, Looper.loop()
6  system_server answers with the app's details            (bindApplication)
7  The app creates the Application, starts the ContentProviders, runs Application.onCreate
8  system_server asks the app to launch the Activity       → onCreate, onStart, onResume
9  The first frame is drawn                                → "Displayed ... +NNNms" in logcat
```
A **cold start** has no process, so all nine steps happen. A **warm start** has a process but no Activity (steps 8 and 9). A **hot start** only resumes an Activity still in memory.

## First half (steps 1 to 5)
- **The tap:** the Launcher is an ordinary app. Its `startActivity` travels through **Binder** to `system_server` (ActivityManager). If the app has no process, `system_server` shows a **starting window** straight away, built from the theme's `windowBackground` and icon. That is why a start can *feel* instant, and why that theme matters.
- **The fork:** `system_server` writes to the **Zygote socket**; Zygote calls `fork()`. The child shares the preloaded framework (copy-on-write) and **specialises itself**: it changes its user ID to the app's own (`u0_a125` is Linux user 10125), which is the start of the app sandbox. This takes milliseconds because no JVM is started from scratch.
- **`ActivityThread.main()`** is the app's real `main`. It (1) `Looper.prepareMainLooper()`, makes the main thread's **mailbox**; (2) `attach`, calls `system_server` over Binder: "I'm alive"; (3) `Looper.loop()`, an endless loop: wait for a message, run it, repeat. From here on, everything (lifecycle, clicks, drawing) arrives as a message on the main thread. One slow message makes the next ones wait: a dropped frame, and eventually an ANR.

## Second half (steps 6 to 9)
- **`bindApplication`:** arrives on a **Binder thread**, which only **posts a message** into the main Looper. The main thread does the work. The standard handoff: *Binder thread → main thread*.
- **The order that decides cold start** (all on the main thread):
  ```text
  1. Application object created      (constructor)
  2. attachBaseContext(...)
  3. every ContentProvider.onCreate() (yours AND every library's)
  4. Application.onCreate()           (your code)
  ```
  Providers run **before** your `Application.onCreate`. Many libraries used a ContentProvider to initialise themselves, and each one added startup time before your own code even began. Jetpack **App Startup** replaces many providers with one that initialises in dependency order.
- **The Activity:** another message on the main Looper. `onCreate` (where `setContent { }` or layout inflation happens), `onStart`, `onResume`.
- **The first frame:** at the next VSYNC the system measures, lays out and draws; `system_server` then logs `Displayed pkg/.Activity: +NNNms`.

## The rule
> **Cold-start time is how long the main thread is busy before the first frame.** So: lighter providers, a lighter `Application.onCreate`, a lighter `Activity.onCreate`; defer what can wait; move the rest to a background thread.

## MQ31 and MQ32
**MQ31 · How does tapping an app icon launch the app?** The nine steps above.
**MQ32 · How does the Launcher know the installed apps?** `PackageManager` in `system_server` scanned every APK at boot (the `pms_*` events timed in DD-02) and keeps the list, updated on every install and removal. The Launcher asks for every activity that declares `MAIN` + `LAUNCHER` (through the `LauncherApps` service or `queryIntentActivities`) and gets a callback when an app is installed or removed.
✏️ Write both in my own words (2 lines each).

## The commands, explained
```bash
adb shell am start -S -W dev.builtbyswap.kaizen/.MainActivity
```
| Part | Meaning |
|---|---|
| `adb shell` | Run the command **on the phone or emulator** |
| `am` | **Activity Manager** tool: talks to the service that starts apps |
| `start` | Start an Activity |
| `-S` | **S**top the app first, which forces a **cold** start |
| `-W` | **W**ait for the launch to finish, then print `LaunchState` and `TotalTime` |
| `package/.Class` | Which Activity (the dot means "inside the app's own package") |

Without `-S` an app that is already running just comes to the front (a warm or hot start), so the timing means something else. Without `-W` the command returns at once.

## My data: the fork, from my emulator

In [ ]:
ps = """
u0_a125      19878  1047   15681904 188160 0                   0 S dev.builtbyswap.kaizen
root          1047     1   15363768  80808 0                   0 S zygote64
"""

rows = [line.split() for line in ps.strip().splitlines()]
procs = {int(r[1]): (int(r[2]), r[-1], r[0]) for r in rows}          # pid -> (parent pid, name, user)

for pid, (ppid, name, user) in procs.items():
    print(f"{name}: PID {pid}, parent PID {ppid}, user {user}")

app_pid = next(pid for pid, (_, name, _) in procs.items() if name.endswith("kaizen"))
zygote_pid = next(pid for pid, (_, name, _) in procs.items() if name == "zygote64")
assert procs[app_pid][0] == zygote_pid                                 # the app's parent is Zygote
print("the app's parent is zygote64 (PID", zygote_pid, "): the app was forked from Zygote")

uid = 10000 + int(procs[app_pid][2].split("_a")[1])                    # u0_a125 -> 10125
print("Linux user id of the app:", uid)
assert uid == 10125

## My data: the cold-start time
`am start -S -W` gave `TotalTime: 294`, then `289`. The `Displayed dev.builtbyswap.kaizen/.MainActivity for user 0: +289ms` line in logcat (written by `ActivityTaskManager`, not by the app: the system times the launch from outside) matched the second run exactly: two tools reporting one moment.

**One run is not a measurement.** Startup time varies with caches, CPU frequency and background work. Take **five runs and use the median**, and compare future changes (a new library, an SDK, a Baseline Profile) against that baseline:
```bash
for i in 1 2 3 4 5; do adb shell am start -S -W dev.builtbyswap.kaizen/.MainActivity | grep TotalTime; done
```

In [ ]:
from statistics import median

def summarize(runs_ms):
    return {"runs": len(runs_ms), "median": median(runs_ms), "min": min(runs_ms), "max": max(runs_ms)}

def regression(baseline_ms, new_ms, allowed=0.10):
    """True if the new build is more than 10% slower than the baseline."""
    return new_ms > baseline_ms * (1 + allowed)

mine = [294, 289]                        # my two runs; add the other three when measured
baseline = summarize(mine)
print(baseline)
print("a 330 ms build is a regression:", regression(baseline["median"], 330))
print("a 300 ms build is a regression:", regression(baseline["median"], 300))
assert baseline["median"] == 291.5
assert regression(baseline["median"], 330) is True and regression(baseline["median"], 300) is False

## Still to do
**Perfetto trace of the cold start** (no new project needed): record 10 seconds with `adb shell perfetto -o /data/misc/perfetto-traces/cold.pftrace -t 10s sched freq idle am wm gfx view binder_driver hal dalvik input res memory`, start the app in a second tab, `adb pull` the file, open it at ui.perfetto.dev. Search for `Start proc`, `bindApplication`, `activityStart` and `Choreographer#doFrame`; fill in:

| Landmark | Starts at (ms) | Lasts (ms) |
|---|---|---|
| Start proc | 0 | |
| bindApplication | | |
| activityStart | | |
| first Choreographer#doFrame | | |

**The probe lab** (a throwaway Android project, never pushed): an `Application` with log lines in its constructor, `attachBaseContext` and `onCreate`, a `ContentProvider` that logs in `onCreate`, and an Activity that logs in `onCreate`, `onStart` and `onResume`. Expected: seven `Probe` lines in the order *constructor → attachBaseContext → provider → Application.onCreate → Activity.onCreate → onStart → onResume*, all on the thread `main`.

## Bug diary
| Error or symptom | Cause | Fix |
|---|---|---|
| I called one run a measurement | Startup time varies from run to run (294 then 289 ms) | Five runs, take the median |

## Quiz and mistakes
Right: every `adb` flag explained back to me (`-S`, `-W`); the PPID check proved the fork. No quiz today.

## English and thinking
Words: **overhead** (extra cost), **redundant** (extra; unnecessary, or kept for safety). Grammar: past simple vs present perfect. Thinking model: **trade-off matrix**.
✏️ Three sentences with both words. ✏️ Drill: a 3×4 matrix for one decision from today (for example how to initialise an SDK: eagerly in `Application.onCreate`, lazily on first use, on a background thread, or with App Startup, scored against cold-start time, simplicity, correctness and testability).

## In my own words
✏️ Why does a ContentProvider add to cold-start time even though my own code has not started yet?

## Next
**AI-03** continues on Sunday (the backward pass step by step). **PRJ-03:** Postgres.